In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("orders (1).csv")

In [3]:
msi_df = df[df["Product"] == "MIS"].reset_index(drop=True)

In [4]:
msi_df

,Time,Type,Instrument,Product,Qty.,Avg. price,Status
0,16/12/21 15:08,BUY,ASHOKLEY,MIS,1000/1000,125.70,COMPLETE
1,16/12/21 14:13,BUY,TATAMOTORS,MIS,250/250,490.55,COMPLETE
2,16/12/21 13:54,BUY,ASHOKLEY,MIS,0/1000,127.10,CANCELLED
3,16/12/21 13:21,SELL,TATAMOTORS,MIS,250/250,492.10,COMPLETE
4,16/12/21 12:51,BUY,TATAMOTORS,MIS,0/250,490.80,CANCELLED
5,16/12/21 12:39,SELL,ASHOKLEY,MIS,1000/1000,125.96,COMPLETE
6,16/12/21 12:29,BUY,ASHOKLEY,MIS,2000/2000,125.70,COMPLETE
7,16/12/21 11:22,SELL,ASHOKLEY,MIS,2000/2000,125.95,COMPLETE
8,16/12/21 10:47,SELL,ASHOKLEY,MIS,0/2000,124.45,CANCELLED
9,16/12/21 10:46,SELL,ASHOKLEY,MIS,2000/2000,125.95,COMPLETE


In [5]:
msi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Time        12 non-null     object 
 1   Type        12 non-null     object 
 2   Instrument  12 non-null     object 
 3   Product     12 non-null     object 
 4   Qty.        12 non-null     object 
 5   Avg. price  12 non-null     float64
 6   Status      12 non-null     object 
dtypes: float64(1), object(6)
memory usage: 804.0+ bytes


In [6]:
msi_df["Qty."] = msi_df["Qty."].str.split('/').str[0].astype(int)

In [7]:
msi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Time        12 non-null     object 
 1   Type        12 non-null     object 
 2   Instrument  12 non-null     object 
 3   Product     12 non-null     object 
 4   Qty.        12 non-null     int32  
 5   Avg. price  12 non-null     float64
 6   Status      12 non-null     object 
dtypes: float64(1), int32(1), object(5)
memory usage: 756.0+ bytes


In [8]:
msi_df["Turnover"] = msi_df["Qty."] * msi_df["Avg. price"]

In [9]:
def brokerage_cal(turnover):
    x = (turnover * (0.03 / 100))
    if x >= 20:
        x = 20
        return x
    else:
        return x

In [10]:
msi_df["Brokerage"] = msi_df["Turnover"].apply(brokerage_cal)

In [11]:
def STT_cal(row):
    if row['Type'] == "SELL":
        return round((row["Turnover"] * (0.025 / 100)),2)
    else:
        return 0

In [12]:
msi_df["STT/CTT"] = msi_df[["Type","Turnover"]].apply(STT_cal,axis=1)

In [13]:
msi_df["ETC"]=round((msi_df["Turnover"] * (0.00307 / 100)),2)

In [14]:
msi_df["SEBI"] =round(msi_df["Turnover"] * (10 / 10000000),2)

In [15]:
def GST_cal(row):
    result = round(row.sum() * (18 / 100),2)
    return result

In [16]:
msi_df["GST"] = msi_df[["Brokerage","SEBI","ETC"]].apply(GST_cal,axis=1)

In [17]:
msi_df["Stamp Charges"] = msi_df["Turnover"].where(msi_df["Type"] == "BUY",0).mul(0.003 / 100).round(2)

In [18]:
msi_df["Total Charges"] = msi_df.loc[:,"Brokerage":"Stamp Charges"].sum(axis=1)

In [19]:
msi_df

,Time,Type,Instrument,Product,Qty.,Avg. price,Status,Turnover,Brokerage,STT/CTT,ETC,SEBI,GST,Stamp Charges,Total Charges
0,16/12/21 15:08,BUY,ASHOKLEY,MIS,1000,125.70,COMPLETE,125700.0,20.0,0.00,3.86,0.13,4.32,3.77,32.08
1,16/12/21 14:13,BUY,TATAMOTORS,MIS,250,490.55,COMPLETE,122637.5,20.0,0.00,3.76,0.12,4.30,3.68,31.86
2,16/12/21 13:54,BUY,ASHOKLEY,MIS,0,127.10,CANCELLED,0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00
3,16/12/21 13:21,SELL,TATAMOTORS,MIS,250,492.10,COMPLETE,123025.0,20.0,30.76,3.78,0.12,4.30,0.00,58.96
4,16/12/21 12:51,BUY,TATAMOTORS,MIS,0,490.80,CANCELLED,0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00
5,16/12/21 12:39,SELL,ASHOKLEY,MIS,1000,125.96,COMPLETE,125960.0,20.0,31.49,3.87,0.13,4.32,0.00,59.81
6,16/12/21 12:29,BUY,ASHOKLEY,MIS,2000,125.70,COMPLETE,251400.0,20.0,0.00,7.72,0.25,5.03,7.54,40.54
7,16/12/21 11:22,SELL,ASHOKLEY,MIS,2000,125.95,COMPLETE,251900.0,20.0,62.98,7.73,0.25,5.04,0.00,96.00
8,16/12/21 10:47,SELL,ASHOKLEY,MIS,0,124.45,CANCELLED,0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00
9,16/12/21 10:46,SELL,ASHOKLEY,MIS,2000,125.95,COMPLETE,251900.0,20.0,62.98,7.73,0.25,5.04,0.00,96.00


In [20]:
stock_type_wise_summary = (msi_df.groupby(["Instrument", "Type"]).sum(numeric_only=True).reset_index())

In [21]:
stock_type_wise_summary

,Instrument,Type,Qty.,Avg. price,Turnover,Brokerage,STT/CTT,ETC,SEBI,GST,Stamp Charges,Total Charges
0,ASHOKLEY,BUY,5000,629.75,628350.0,80.0,0.00,19.30,0.64,17.99,18.85,136.78
1,ASHOKLEY,SELL,5000,502.31,629760.0,60.0,157.45,19.33,0.63,14.40,0.00,251.81
2,TATAMOTORS,BUY,250,981.35,122637.5,20.0,0.00,3.76,0.12,4.30,3.68,31.86
3,TATAMOTORS,SELL,250,492.10,123025.0,20.0,30.76,3.78,0.12,4.30,0.00,58.96


In [22]:
summary = msi_df.pivot_table(
    index = "Instrument",
    columns = "Type",
    values = ["Turnover","Total Charges"],
    aggfunc="sum"
)
summary.columns = [f"{value}_{tran_type}" for value, tran_type in summary.columns]
summary.reset_index(inplace=True)

In [23]:
summary["Gross PnL"] = summary["Turnover_SELL"] - summary["Turnover_BUY"]

In [24]:
summary["Total charges"] = summary["Total Charges_BUY"] + summary["Total Charges_SELL"]

In [25]:
summary["Net PnL"] = summary["Gross PnL"] - summary["Total charges"]

In [26]:
summary["% Charge"]= (summary["Total charges"] / summary["Gross PnL"] * 100)

In [27]:
stockwise_summary = summary[["Instrument","Gross PnL","Total charges","Net PnL","% Charge"]].round(2)

In [28]:
grand_total = {
    "Instrument" : "Total",
    "Gross PnL" : stockwise_summary["Gross PnL"].sum(),
    "Total charges" : stockwise_summary["Total charges"].sum(),
    "Net PnL" : stockwise_summary["Net PnL"].sum(),
    "% Charge" : round((stockwise_summary["Total charges"].sum() / stockwise_summary["Gross PnL"].sum() * 100),2)
}

In [29]:
stockwise_summary.loc[len(stockwise_summary)] = grand_total

In [30]:
stockwise_summary

,Instrument,Gross PnL,Total charges,Net PnL,% Charge
0,ASHOKLEY,1410.0,388.59,1021.41,27.56
1,TATAMOTORS,387.5,90.82,296.68,23.44
2,Total,1797.5,479.41,1318.09,26.67


In [31]:
writer = pd.ExcelWriter("Kite_PnL.xlsx",engine="openpyxl")
msi_df.to_excel(writer,sheet_name="Charges On Individual trade",index=False)
stock_type_wise_summary.to_excel(writer,sheet_name="Stock & Type Wise Summary",index=False)
stockwise_summary.to_excel(writer,sheet_name="Stock Wise Summary",index=False)

In [32]:
writer.close()